# Zagent — 三算子自改进闭环（Notebook 入口）

在 Jupyter 里跑 Data-RSI / Harness-RSI / Model-RSI。**只需改第 1 个 cell 的路径，然后 Cell → Run All。**

前提：（含 numpy/pandas/scipy/pyarrow/torch）。

In [ ]:
# ===== 1. 配置（只改这里）=====
PKG = r""   # 数据包目录：含 pkg_api.py（Model-RSI 用）

# Model-RSI 快速评估参数（CPU 冒烟，秒级）
SETTINGS = {
    "iterations": 6,
    "blocks": 2,
    "max_train": 8000,
    "fixed_epochs": 10,
    "n_seeds": 2,
    "threads": 2,
    "isolated": True,
}

## 2. 体检数据包
先体检（sha1 对账 + 依赖 + CUDA），不通过就停。

In [ ]:
from zagent.migrate import doctor
import sys
if not PKG:
    print("请先在第 1 个 cell 里填 PKG 路径")
    sys.exit(1)
rc = doctor(PKG)
if rc != 0:
    sys.exit("体检未通过，请先解决上面的 FAIL")

## 3. Model-RSI 闭环
进化 alpha 训练配方，只保留严格更优。

In [ ]:
from zagent.model import run_alpha
from pathlib import Path
stats = run_alpha(Path(PKG), SETTINGS, offline=True)
print("最佳 RankIC:", round(stats.best_rank_ic, 4))
print("最佳配方:", stats.best_recipe)

## 4.（可选）Data-RSI 数据清洗
对 X.npy 做去极值→截面标准化→中位数填充。

In [ ]:
from zagent.data import preprocess_panel, PreprocessConfig
import numpy as np
# X = np.load(r"X.npy")
# out = preprocess_panel(X, PreprocessConfig(), verbose=True)
# np.save(r"X_clean.npy", out)
print("取消注释上面的三行即可用（按需）")